In [ ]:
# Week 3 - KPI Transformation

#This notebook demonstrates the KPI transformation pipeline and validates the intermediate outputs generated from the MES and PLM datasets.

In [5]:
import pandas as pd
from pathlib import Path

LANDING_DIR = Path("../data/landing")
CURATED_DIR = Path("../data/curated")

logs = pd.read_parquet(LANDING_DIR / "machine_logs.parquet")
orders = pd.read_parquet(LANDING_DIR / "production_orders.parquet")
downtime = pd.read_parquet(LANDING_DIR / "downtime_events.parquet")
reference = pd.read_parquet(LANDING_DIR / "electrolyser_reference.parquet")
kpis = pd.read_parquet(CURATED_DIR / "production_kpis.parquet")

In [6]:
logs.head()

,timestamp,machine_id,status,cycle_time_ms,output_count,defect_flag
0,2026-03-29 20:29:27,M02,Running,1363,4,False
1,2025-09-05 18:16:54,M02,Running,1916,2,False
2,2025-08-03 05:43:05,M01,Running,1247,4,False
3,2026-05-08 12:21:09,M01,Running,2266,11,False
4,2025-11-08 08:03:36,M07,Running,2006,5,False


In [7]:
orders.head()

,order_id,product_id,start_time,end_time,planned_qty,actual_qty
0,ORD-0001,PROD-014,2025-09-29 00:18:35,2025-09-29 07:18:35,244,225
1,ORD-0002,PROD-024,2026-04-06 08:26:23,2026-04-06 12:26:23,215,207
2,ORD-0003,PROD-086,2026-03-24 01:10:20,2026-03-24 06:10:20,268,254
3,ORD-0004,PROD-042,2026-04-30 09:23:19,2026-04-30 14:23:19,489,485
4,ORD-0005,PROD-075,2026-07-09 19:51:43,2026-07-10 02:51:43,291,287


In [8]:
downtime.head()

,event_id,machine_id,start_time,end_time,reason_code
0,DT-001,M10,2025-12-20 11:52:51,2025-12-20 15:16:51,Sensor Fault
1,DT-002,M08,2025-10-23 23:55:32,2025-10-24 01:23:32,Machine Failure
2,DT-003,M03,2026-01-14 00:35:07,2026-01-14 01:30:07,Power Outage
3,DT-004,M10,2026-05-13 06:13:47,2026-05-13 07:16:47,Scheduled Maintenance
4,DT-005,M08,2025-11-05 05:16:33,2025-11-05 07:38:33,Quality Inspection


In [9]:
reference.head()

,product_id,benchmark_efficiency_nm3_per_kwh,rated_power_kw
0,PROD-001,1.86,10
1,PROD-002,1.84,10
2,PROD-003,1.80,20
3,PROD-004,1.74,10
4,PROD-005,1.96,20


In [10]:
kpis.head()

,order_id,product_id,planned_qty,actual_qty,total_output,defects,defect_rate,avg_cycle_time_ms,planned_time_ms,downtime_ms,operating_time_ms,benchmark_efficiency_nm3_per_kwh,rated_power_kw,ideal_cycle_time_ms,performance_factor,quality_factor,availability_factor,oee
0,ORD-0001,PROD-014,244,225,7,0,0.00,2201.00,25200000.0,0.0,25200000.0,1.77,20,101694.915254,0.028249,1.00,1.0,0.028249
1,ORD-0002,PROD-024,215,207,25,0,0.00,1547.00,14400000.0,0.0,14400000.0,1.78,15,134831.460674,0.234082,1.00,1.0,0.234082
2,ORD-0003,PROD-086,268,254,19,0,0.00,1642.50,18000000.0,0.0,18000000.0,1.85,15,129729.729730,0.136937,1.00,1.0,0.136937
3,ORD-0004,PROD-042,489,485,14,0,0.00,1579.75,18000000.0,0.0,18000000.0,1.76,30,68181.818182,0.053030,1.00,1.0,0.053030
4,ORD-0005,PROD-075,291,287,20,1,0.05,1067.00,25200000.0,0.0,25200000.0,1.71,30,70175.438596,0.055695,0.95,1.0,0.052910


In [11]:
print("Total Orders :", len(orders))
print("Total Machine Logs :", len(logs))
print("Total Downtime Events :", len(downtime))

Total Orders : 500
Total Machine Logs : 3000
Total Downtime Events : 300


In [12]:
kpis[[
    "total_output",
    "defects",
    "defect_rate",
    "availability_factor",
    "performance_factor",
    "quality_factor",
    "oee"
]].describe()

,total_output,defects,defect_rate,availability_factor,performance_factor,quality_factor,oee
count,500.000000,500.000000,356.000000,500.000000,500.000000,356.000000,356.000000
mean,12.862000,0.080000,0.010970,0.977550,0.099310,0.989030,0.132906
std,13.048966,0.285943,0.063857,0.094803,0.129332,0.063857,0.129642
min,0.000000,0.000000,0.000000,0.160000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.052910
50%,10.000000,0.000000,0.000000,1.000000,0.060661,1.000000,0.097502
75%,21.000000,0.000000,0.000000,1.000000,0.149292,1.000000,0.172143
max,75.000000,2.000000,1.000000,1.000000,1.236559,1.000000,1.182796


In [13]:
kpis.isnull().sum()

order_id                              0
product_id                            0
planned_qty                           0
actual_qty                            0
total_output                          0
defects                               0
defect_rate                         144
avg_cycle_time_ms                   144
planned_time_ms                       0
downtime_ms                           0
operating_time_ms                     0
benchmark_efficiency_nm3_per_kwh      0
rated_power_kw                        0
ideal_cycle_time_ms                   0
performance_factor                    0
quality_factor                      144
availability_factor                   0
oee                                 144
dtype: int64

In [14]:
## Diagnostic Queries-The following diagnostic queries were executed to validate the transformation process.

# Count orders with zero total output
print("\nOrders with zero total_output:")
print((kpis["total_output"] == 0).sum())

# Check columns containing NULL values
print("\nNULL values per column:")
print(kpis.isnull().sum())

# Inspect sample rows with missing KPI values
print("\nSample orders with zero total_output:")
print(
    kpis[kpis["total_output"] == 0][
        [
            "order_id",
            "product_id",
            "total_output",
            "defect_rate",
            "quality_factor",
            "oee"
        ]
    ].head(10)
)


Orders with zero total_output:
144

NULL values per column:
order_id                              0
product_id                            0
planned_qty                           0
actual_qty                            0
total_output                          0
defects                               0
defect_rate                         144
avg_cycle_time_ms                   144
planned_time_ms                       0
downtime_ms                           0
operating_time_ms                     0
benchmark_efficiency_nm3_per_kwh      0
rated_power_kw                        0
ideal_cycle_time_ms                   0
performance_factor                    0
quality_factor                      144
availability_factor                   0
oee                                 144
dtype: int64

Sample orders with zero total_output:
    order_id product_id  total_output  defect_rate  quality_factor  oee
5   ORD-0006   PROD-081             0          NaN             NaN  NaN
6   ORD-0007   PROD-002